# CPS matched continuation test

**Grand Challenge release notebook.** This self-contained notebook closes the diagnostic loop: measure, plan, preregister, fork, compare, and preserve the complete evidence chain.


## Release contract

- **Scientific question:** Does the CPS-selected short-horizon intervention improve the declared continuation criteria under matched conditions?
- **Default execution path:** Generate a compact packet and recommendation internally, then run baseline and intervention forks from identical weights and batches.
- **Evidence boundary:** One continuation is a mechanism test; multiple checkpoints and seeds are required for a general optimizer claim.
- **Primary outputs:** Diagnostic packet, preregistered recommendation, paired continuation trajectories, summary metrics, and export archive.


## Interpretation checklist

- Verify that the recommendation was written before the continuation fork ran.
- Confirm that both forks used identical initial weights, token budget, and batch order.
- Report both positive and negative intervention results without changing the success criterion.


## Acceptance logic

A successful intervention should improve a preregistered stability or loss criterion without changing token budget. One short continuation is a mechanism test; multiple checkpoints and seeds are needed for a general optimizer claim.

In [ ]:
import os, pathlib, subprocess, sys, time
from IPython.display import Markdown, display

REPO_URL = os.environ.get("CPS_REPO_URL", "https://github.com/fyremael/CPS.git")
GIT_REF = os.environ.get("CPS_GIT_REF", "main")
repo = pathlib.Path("/content/CPS")

print("[BOOT] Preparing the CPS repository", flush=True)
print(f"[BOOT] source={REPO_URL}", flush=True)
print(f"[BOOT] ref={GIT_REF}", flush=True)
if not repo.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", GIT_REF, REPO_URL, str(repo)], check=True)
else:
    subprocess.run(["git", "-C", str(repo), "fetch", "origin", GIT_REF], check=True)
    subprocess.run(["git", "-C", str(repo), "checkout", GIT_REF], check=True)
    subprocess.run(["git", "-C", str(repo), "pull", "--ff-only", "origin", GIT_REF], check=True)
os.chdir(repo)
print("[BOOT] Installing CPS with Pythia and notebook dependencies", flush=True)
subprocess.run([
    sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
    "-e", ".[pythia,notebooks]"
], check=True)
print(f"[BOOT] Ready: {repo}", flush=True)

# Editable installs write a .pth file, but the running Colab kernel does not
# automatically reprocess newly-created .pth files. Put the source tree on
# sys.path explicitly so the very next cell can import CPS without a restart.
import importlib
src_dir = repo / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))
importlib.invalidate_caches()
import cps
print(f"[BOOT] CPS import verified from {cps.__file__}", flush=True)


In [ ]:
from cps.notebook import stage_banner
stage_banner(
    "0",
    "Record the runtime contract",
    objective="Expose the software and accelerator environment before scientific execution.",
    deliverable="A visible runtime inventory for reproducibility.",
)
from cps.notebook import show_environment
runtime = show_environment()

## Stage 1 — obtain the diagnostic packet

The default path is self-contained: run a compact Pythia-70M probe. Set `CPS_EVIDENCE_PATH` only to reuse a deliberate external packet or ZIP archive.


In [ ]:
from cps.notebook import stage_banner
stage_banner('1', 'obtain the diagnostic packet', objective='The default path is selfcontained: run a compact Pythia70M probe. Set CPSEVIDENCEPATH only to reuse a deliberate external packet or ZIP archive.', deliverable="The artifacts and console evidence described in this stage.")

import os
from cps.pythia.notebook_support import load_evidence_packet, run_self_contained_probe

revision = os.environ.get("CPS_REVISION", "step1000")
evidence_path = os.environ.get("CPS_EVIDENCE_PATH")
if evidence_path:
    print(f"[CONTINUATION] Reusing explicit evidence: {evidence_path}", flush=True)
    packet = load_evidence_packet(evidence_path)
else:
    print(f"[CONTINUATION] Running compact probe at revision={revision}", flush=True)
    generated_root = run_self_contained_probe(
        revision=revision,
        run_name="pythia-70m-continuation-plan",
    )
    packet = load_evidence_packet(generated_root)
A = packet.matrix
print(f"[CONTINUATION] diagnostic packet={packet.root}", flush=True)


## Stage 2 — preregister the intervention

Score the declared candidate family and preserve the selected learning-rate scale before either continuation fork is inspected.


In [ ]:
from cps.notebook import stage_banner
stage_banner('2', 'preregister the intervention', objective='Score the declared candidate family and preserve the selected learningrate scale before either continuation fork is inspected.', deliverable="The artifacts and console evidence described in this stage.")

from pathlib import Path
from cps.pythia.notebook_support import select_candidate_edges, write_planner_recommendation
from cps.pythia.planner import damping_family, plan_scalar_control

candidate_scales = [1.0, 0.98, 0.95, 0.90, 0.80]
edges = select_candidate_edges(packet, maximum=8)
recommendation = plan_scalar_control(
    "learning_rate_scale",
    1.0,
    candidate_scales,
    lambda scale: damping_family(A, 1.0 - scale),
    edges,
)
plan_root = Path("/content/cps-artifacts/continuation-plan")
plan_root.mkdir(parents=True, exist_ok=True)
plan_path = write_planner_recommendation(
    plan_root,
    recommendation,
    selected_edges=edges,
    candidate_grid=candidate_scales,
    control_operationalization={
        "surrogate_family": "isotropic_damping",
        "translation": "gamma = 1 - learning_rate_scale",
        "warning": "The matched continuation is the actual intervention test.",
    },
)
print(f"[CONTINUATION] preregistered recommendation={plan_path}", flush=True)
print(f"[CONTINUATION] planned learning-rate scale={recommendation.recommended}", flush=True)


## Stage 3 — resolve the continuation contract

Environment variables allow the Colab CLI to inject the checkpoint, number of steps, and selected control without editing the notebook.

In [ ]:
from cps.notebook import stage_banner
stage_banner('3', 'resolve the continuation contract', objective='Environment variables allow the Colab CLI to inject the checkpoint, number of steps, and selected control without editing the notebook.', deliverable="The artifacts and console evidence described in this stage.")

import os
from cps.notebook import show_config
from cps.pythia.continuation import ContinuationConfig, ContinuationControl

config = ContinuationConfig(
    revision=revision,
    steps=int(os.environ.get("CPS_CONTINUATION_STEPS", "20")),
    intervention=ContinuationControl(
        name="cps",
        learning_rate_scale=float(os.environ.get("CPS_LR_SCALE", str(recommendation.recommended))),
        beta1=float(os.environ["CPS_BETA1"]) if "CPS_BETA1" in os.environ else None,
        beta2=float(os.environ["CPS_BETA2"]) if "CPS_BETA2" in os.environ else None,
        epsilon_scale=float(os.environ.get("CPS_EPSILON_SCALE", "1.0")),
        gradient_clip=float(os.environ["CPS_GRADIENT_CLIP"]) if "CPS_GRADIENT_CLIP" in os.environ else None,
    ),
    output_dir="/content/cps-artifacts/continuation",
    verbose=True,
)
show_config(config)

## Stage 4 — run both forks

The live log reports loss and gradient norm at every step for each fork. This makes divergence, stalls, or numerical spikes visible in remote `colab-cli log` output.

In [ ]:
from cps.notebook import stage_banner
stage_banner('4', 'run both forks', objective='The live log reports loss and gradient norm at every step for each fork. This makes divergence, stalls, or numerical spikes visible in remote colabcli log output.', deliverable="The artifacts and console evidence described in this stage.")

from cps.pythia.continuation import run_matched_continuation

result_path = run_matched_continuation(config)
print(f"[CONTINUATION] evidence={result_path}", flush=True)

## Stage 5 — compare trajectories

The plot is paired by construction: step *k* in both forks used the same token batch.

In [ ]:
from cps.notebook import stage_banner
stage_banner('5', 'compare trajectories', objective='The plot is paired by construction: step k in both forks used the same token batch.', deliverable="The artifacts and console evidence described in this stage.")

import json, pathlib, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display

payload=json.loads(pathlib.Path(result_path).read_text())
frames=[]
summary_rows=[]
for name,result in payload["results"].items():
    frame=pd.DataFrame(result["records"])
    frame["fork"]=name
    frames.append(frame)
    summary_rows.append({
        "fork": name,
        "final loss": result["final_loss"],
        "maximum gradient norm": result["maximum_gradient_norm"],
        "loss spike": result["loss_spike"],
    })
trajectory=pd.concat(frames, ignore_index=True)
display(pd.DataFrame(summary_rows))
axis=trajectory.pivot(index="step", columns="fork", values="loss").plot(marker="o", figsize=(10,5))
axis.set_title("Matched continuation loss")
axis.set_ylabel("loss")
axis.grid(True, alpha=0.25)
plt.tight_layout(); plt.show()
axis=trajectory.pivot(index="step", columns="fork", values="gradient_norm").plot(figsize=(10,5))
axis.set_title("Matched continuation gradient norm")
axis.set_ylabel("||g||₂")
axis.grid(True, alpha=0.25)
plt.tight_layout(); plt.show()

## Final stage — export the evidence packet

Every release notebook ends with the same preservation step. The archive contains the evidence produced in this runtime and is suitable for Colab CLI retrieval or manual download.


In [ ]:
from cps.notebook import export_artifacts, stage_banner

stage_banner(
    "EXPORT",
    "Package the evidence",
    objective="Collect the run artifacts into one portable archive.",
    deliverable="/content/cps-export.zip",
)
archive = export_artifacts()
print(f"[EXPORT] archive={archive}", flush=True)
